<a href="https://colab.research.google.com/github/237marek9-per/Problem_poctu_nepozorovanych_druhov/blob/main/Nietzche.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nietzsche — extrakcia a porovnanie slovníkov

## Pipeline

1. **Extrakcia textu**: `pdftotext -layout` (poppler)
2. **Čistenie**: hlavičky, dátumy, čísla, URL
3. **Spojenie delených slov** (`deutsch-\nfranzösischen` → `deutschfranzösischen`)
4. **Apostrof**: `Zarathustra's` → `Zarathustras`
5. **Sperrsatz**: `t r o t z` → `trotz` (min 3 jednopísmenkové tokeny)
6. **Normalizácia**: lowercase + `ß` → `ss`
7. **Filter** krátkych tokenov (`< 2 znaky`)


Pred filtrom: 25 unikátnych jednopísmenkových tokenov, 475 výskytov.  
Po filtri `len >= 2`: **0 jednopísmenkových**


Filter `len >= 2` je pre nemčinu štandardný. Nemčina nemá jednopísmenkové slová (`a`, `o`, `i` v latinskom zmysle neexistujú). Jediné jednopísmenkové slova ktoré v Nietzschem nájdeš sú:
- iniciály mien (`A. W. Schlegel`)
- bibliografické odkazy (`p. 23`, `I. p. 12`)
- 2-písmenové slová zdôraznené separaciou (`s o`, `o b`, `d u`)
- citácie písmen (`A und O` ako alfa-omega)
- typografické zvyšky

Žiadny z týchto prípadov nie je významonosné slovo, ktoré by malo byť v slovníku.


#PROGRAM


In [5]:
# poppler-utils v Colabe je predinštalovaný
!apt-get install -y poppler-utils -q 2>&1 | tail -1

import re
import os
import subprocess
from collections import Counter
import pandas as pd


def extrahuj_text(cesta_k_pdf, od_strany=1, do_strany=None):
    """pdftotext -layout: zachováva Sperrsatz na jednom riadku."""
    cmd = ['pdftotext', '-layout', '-f', str(od_strany)]
    if do_strany is not None:
        cmd += ['-l', str(do_strany)]
    cmd += [cesta_k_pdf, '-']
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
        return result.stdout
    except Exception as e:
        print(f"  Chyba pri {cesta_k_pdf}: {e}")
        return ''


def vycisti_text(text):
    """Kompletné čistenie textu."""
    # --- hlavičky a paratexty ---
    text = re.sub(r'http[s]?://\S+', '', text)
    text = re.sub(r'www\.nietzschesource\.org\S+', '', text)
    text = re.sub(r'Nietzsche Source.*?eKGWB', '', text, flags=re.IGNORECASE)
    text = re.sub(r'Digitale Kritische Gesamtausgabe.*?\(eKGWB\)', '', text, flags=re.IGNORECASE)
    text = re.sub(r'Der Wille zur Macht, by Friedrich Nietzsche, a Project Gutenberg eBook\.', '', text)
    text = re.sub(r'\d{1,2}\.\s\d{1,2}\.\s\d{4}', '', text)
    text = re.sub(r'\d{2}:\d{2}', '', text)
    text = re.sub(r'\b\d+/\d+\b', '', text)
    text = re.sub(r'\b\d+\b', '', text)

    # --- spojenie slov delených pomlčkou na konci riadku ---
    text = re.sub(r'-\s*\n\s*', '', text)

    # --- odstránenie apostrofov ---
    text = re.sub(r"([a-zA-ZäöüÄÖÜß])['\u2019]s\b", r"\1s", text)
    text = re.sub(r"([a-zA-ZäöüÄÖÜß])['\u2019]\s", r"\1 ", text)
    text = re.sub(r"([a-zA-ZäöüÄÖÜß])['\u2019]([a-zA-ZäöüÄÖÜß])", r"\1\2", text)

    # --- Sperrsatz: 't r o t z' -> 'trotz' ---
    def zluc_sperrsatz(match):
        slovo = re.sub(r'\s+', '', match.group(0))
        if len(slovo) >= 3:
            return ' ' + slovo + ' '
        return match.group(0)

    text = re.sub(
        r'(?:(?<=\s)|(?<=^))[a-zA-ZäöüÄÖÜß](?: +[a-zA-ZäöüÄÖÜß]){2,}(?= |$|[\.,;:?!„"\n])',
        zluc_sperrsatz,
        text
    )

    return text


def normalizuj_slovo(slovo):
    """lowercase + ß -> ss"""
    return slovo.lower().replace('ß', 'ss')


def extrahuj_slova(text, min_dlzka=2):
    """Extrahuje slová bez lematizácie."""
    slova_raw = re.findall(r'\b[a-zA-ZäöüÄÖÜß]+\b', text)
    return [normalizuj_slovo(s) for s in slova_raw if len(s) >= min_dlzka]


def spracuj_priecinok(nazov_priecinka, od_strany=1, do_strany=None, min_dlzka=2):
    """Spracuje všetky PDF v priečinku."""
    vsetky_slova = []

    if not os.path.exists(nazov_priecinka):
        print(f"Chyba: Priečinok '{nazov_priecinka}' neexistuje.")
        return

    pdf_subory = sorted(f for f in os.listdir(nazov_priecinka) if f.lower().endswith('.pdf'))
    print(f"Našiel som {len(pdf_subory)} súborov v '{nazov_priecinka}'.")
    rozsah = f"od {od_strany}" + (f" do {do_strany}" if do_strany else " do konca")
    print(f"Strany: {rozsah}")
    print(f"Min dĺžka slova: {min_dlzka}")

    for pdf_meno in pdf_subory:
        cesta = os.path.join(nazov_priecinka, pdf_meno)
        text = extrahuj_text(cesta, od_strany=od_strany, do_strany=do_strany)
        if not text:
            print(f"  ⚠ {pdf_meno}: prázdny text")
            continue

        text = vycisti_text(text)
        slova = extrahuj_slova(text, min_dlzka=min_dlzka)
        vsetky_slova.extend(slova)

        print(f"  ✓ {pdf_meno}: {len(slova)} slov")

    cesta_vsetky = os.path.join(nazov_priecinka, 'vsetky_slova_vysledok.txt')
    cesta_unikatne = os.path.join(nazov_priecinka, 'unikatne_slova_vysledok.txt')

    with open(cesta_vsetky, 'w', encoding='utf-8') as f1:
        f1.write(', '.join(vsetky_slova))

    pocetnost = Counter(vsetky_slova)

    with open(cesta_unikatne, 'w', encoding='utf-8') as f2:
        for slovo, pocet in pocetnost.most_common():
            f2.write(f"{slovo}, {pocet}\n")

    print(f"\n--- HOTOVO ---")
    print(f"Celkovo slov: {len(vsetky_slova)}")
    print(f"Unikátnych: {len(pocetnost)}")

    return pocetnost


print("Funkcie pripravené.")

0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
Funkcie pripravené.


#SPUSTENIE


In [2]:
_ = spracuj_priecinok('1-18')
_ = spracuj_priecinok('VolaKMoci')

Našiel som 18 súborov v '1-18'.
Strany: od 1 do konca
Min dĺžka slova: 2
  ✓ N1.pdf: 41658 slov
  ✓ N10.pdf: 1067 slov
  ✓ N11.pdf: 75793 slov
  ✓ N12.pdf: 19504 slov
  ✓ N13.pdf: 18669 slov
  ✓ N14.pdf: 21667 slov
  ✓ N15.pdf: 57540 slov
  ✓ N16.pdf: 45246 slov
  ✓ N17.pdf: 11288 slov
  ✓ N18.pdf: 24810 slov
  ✓ N2.pdf: 24608 slov
  ✓ N3.pdf: 26877 slov
  ✓ N4.pdf: 28341 slov
  ✓ N5.pdf: 24050 slov
  ✓ N6.pdf: 542 slov
  ✓ N7.pdf: 83683 slov
  ✓ N8.pdf: 75834 slov
  ✓ N9.pdf: 79124 slov

--- HOTOVO ---
Celkovo slov: 660301
Unikátnych: 45894
Našiel som 1 súborov v 'VolaKMoci'.
Strany: od 1 do konca
Min dĺžka slova: 2
  ✓ Der Wille zur Macht, by Friedrich Nietzsche, a Project Gutenberg eBook_.pdf: 95900 slov

--- HOTOVO ---
Celkovo slov: 95900
Unikátnych: 12480


#POROVNANIE

In [4]:
def naciitaj_slova(cesta):
    """Načíta slovník zo súboru.

    keep_default_na=False: pandas štandardne konvertuje slová ako 'null', 'na',
    'NaN', 'None' na chýbajúce hodnoty. Pre slovník to je chyba — 'na' je
    nemecké citoslovce a tieto slová by sa stratili.
    """
    if os.path.exists(cesta):
        df = pd.read_csv(cesta, header=None, names=['slovo', 'pocet'],
                         skipinitialspace=True,
                         keep_default_na=False, na_values=[])
        return set(df['slovo'].astype(str).str.lower().str.strip())
    print(f"Súbor {cesta} neexistuje!")
    return set()


def porovnaj_slovniky(cesta_BD, cesta_AD, vystupny_subor):
    nice_BD = naciitaj_slova(cesta_BD)
    nice_AD = naciitaj_slova(cesta_AD)

    # Odfiltrujem prázdne stringy ktoré môžu vzniknúť pri parsovaní
    nice_BD.discard('')
    nice_AD.discard('')

    nove = nice_AD - nice_BD
    df_v = pd.DataFrame(sorted(list(nove)), columns=['Nove_Slova'])

    print(f"BD ({cesta_BD}): {len(nice_BD)}")
    print(f"AD ({cesta_AD}): {len(nice_AD)}")
    print(f"Prienik (slová v oboch): {len(nice_BD & nice_AD)}")
    print("-" * 40)
    print(f"NOVÝCH (AD − BD): {len(df_v)}")
    print("-" * 40)
    df_v.to_csv(vystupny_subor, index=False, header=False)
    print(f"Uložené: {vystupny_subor}")

    # Sanity check symetrie
    assert len(nice_AD) == len(nice_BD & nice_AD) + len(nove), "Logická chyba!"

    return nice_BD, nice_AD, nove


BD, AD, nove = porovnaj_slovniky(
    '1-18/unikatne_slova_vysledok.txt',
    'VolaKMoci/unikatne_slova_vysledok.txt',
    'rozdiel_slov.txt'
)

BD (1-18/unikatne_slova_vysledok.txt): 45894
AD (VolaKMoci/unikatne_slova_vysledok.txt): 12480
Prienik (slová v oboch): 8593
----------------------------------------
NOVÝCH (AD − BD): 3887
----------------------------------------
Uložené: rozdiel_slov.txt
